# Anomaly Detection using Autoencoder (MNIST)
**Idea:** train an autoencoder only on *normal* data (digit 0). It learns to rebuild normal data well, so normal samples get a **low reconstruction error**. Anomalies (digit 1) are rebuilt badly, so they get a **high error** and are flagged.

Steps: a) Import libraries, b) Load dataset, c) Encoder, d) Decoder, e) Compile, then train and detect anomalies.

## a. Import required libraries
- `numpy`: arrays and maths (mean, square, percentile)
- `matplotlib`: plots and images
- `tensorflow` / `keras`: building and training the neural network

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

## b. Upload / access the dataset
MNIST (28x28 grayscale digit images) is available directly through Keras, so no manual upload is needed.
- Pixels are scaled from 0-255 to 0-1 (helps training and matches the sigmoid output).
- Digit **0** = normal data (used for training), digit **1** = anomaly (used only for testing).
- Each image is flattened from 28x28 to 784 values because Dense layers need 1-D input.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print("Training data shape:", x_train.shape)
print("Testing data shape:", x_test.shape)

# Normalize to 0-1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Normal = 0, Anomaly = 1
x_train_normal = x_train[y_train == 0].reshape(-1, 784)
x_test_normal = x_test[y_test == 0].reshape(-1, 784)
x_test_anomaly = x_test[y_test == 1].reshape(-1, 784)

print("Normal training samples:", len(x_train_normal))
print("Normal testing samples:", len(x_test_normal))
print("Anomaly samples:", len(x_test_anomaly))

## c. Encoder: converts input into a latent representation
The encoder compresses each 784-pixel image into just **32 numbers** (the latent representation / bottleneck):

`784 -> 128 -> 64 -> 32` (Dense layers with ReLU).

The small bottleneck forces the network to keep only the most important features of a zero.

In [ ]:
input_dim = 784
latent_dim = 32

encoder = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(latent_dim, activation="relu")
], name="encoder")

## d. Decoder: converts the latent representation back to the original input
The decoder mirrors the encoder and rebuilds the image from the 32-number code:

`32 -> 64 -> 128 -> 784`

The last layer uses **sigmoid** so outputs lie in 0-1, the same range as the normalized pixels.
Encoder and decoder are then joined into one autoencoder model: Input -> Encoder -> Decoder -> Output.

In [ ]:
decoder = models.Sequential([
    layers.Input(shape=(latent_dim,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(input_dim, activation="sigmoid")
], name="decoder")

# Combine encoder + decoder
inputs = layers.Input(shape=(input_dim,))
autoencoder = models.Model(inputs, decoder(encoder(inputs)), name="autoencoder")

## e. Compile the model with Optimizer, Loss and Evaluation Metrics
- **Optimizer:** `adam`: adaptive algorithm that updates the weights.
- **Loss:** `mse` (Mean Squared Error): difference between original and reconstructed image; this is minimized.
- **Metric:** `mae` (Mean Absolute Error): extra measure to monitor reconstruction quality.

`summary()` shows each layer, its output shape and number of parameters.

In [ ]:
autoencoder.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

autoencoder.summary()

## Train the autoencoder
Input and target are the **same** (`x_train_normal`), since the model learns to reproduce its input (unsupervised).
- `epochs=20`: 20 passes over the data
- `batch_size=128`: weights updated every 128 images
- `validation_split=0.2`: 20% held out to check performance on unseen data

In [ ]:
history = autoencoder.fit(
    x_train_normal, x_train_normal,
    epochs=20,
    batch_size=128,
    validation_split=0.2,
    shuffle=True
)

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Autoencoder Training Loss")
plt.legend()
plt.show()

## Detect anomalies using reconstruction error
1. Reconstruct normal test images and compute the error per image: mean of (original - reconstructed)^2.
2. Set the **threshold** at the 95th percentile of normal errors (95% of normal images fall below it).
3. Reconstruct the anomaly images (digit 1) and compute their errors.
4. Any image with error **above the threshold** is classified as an anomaly.

In [ ]:
normal_recon = autoencoder.predict(x_test_normal, verbose=0)
normal_errors = np.mean(np.square(x_test_normal - normal_recon), axis=1)

threshold = np.percentile(normal_errors, 95)
print("Anomaly Threshold:", threshold)

anomaly_recon = autoencoder.predict(x_test_anomaly, verbose=0)
anomaly_errors = np.mean(np.square(x_test_anomaly - anomaly_recon), axis=1)

false_positives = np.sum(normal_errors > threshold)
detected = np.sum(anomaly_errors > threshold)

print("\n========== RESULTS ==========")
print("Normal samples wrongly flagged:", false_positives)
print("Anomalies detected:", detected, "out of", len(anomaly_errors))
print("Anomaly Detection Rate:", round(detected / len(anomaly_errors) * 100, 2), "%")
print("False Positive Rate:", round(false_positives / len(normal_errors) * 100, 2), "%")

## Visualize the results
- **Histogram:** normal errors cluster at low values, anomaly errors lie mostly beyond the dashed threshold line.
- **Images:** top row = original anomalies (digit 1), bottom row = reconstructions. They look distorted because the model only learned to draw zeros.

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(normal_errors, bins=50, alpha=0.6, label="Normal")
plt.hist(anomaly_errors, bins=50, alpha=0.6, label="Anomaly")
plt.axvline(threshold, linestyle="--", color="red", label="Threshold")
plt.xlabel("Reconstruction Error")
plt.ylabel("Number of Samples")
plt.title("Normal vs Anomaly Reconstruction Error")
plt.legend()
plt.show()

n = 5
plt.figure(figsize=(12, 5))
for i in range(n):
    plt.subplot(2, n, i + 1)
    plt.imshow(x_test_anomaly[i].reshape(28, 28), cmap="gray")
    plt.title("Original")
    plt.axis("off")

    plt.subplot(2, n, i + 1 + n)
    plt.imshow(anomaly_recon[i].reshape(28, 28), cmap="gray")
    plt.title("Reconstructed")
    plt.axis("off")
plt.tight_layout()
plt.show()

## Conclusion
The autoencoder, trained only on digit 0, reconstructs zeros with low error but reconstructs digit 1 poorly. Using the 95th-percentile threshold, almost all anomalies are detected (about 99.8% in the earlier run) with roughly 5% false positives on normal data.